In [70]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score

from sklearn.linear_model import LassoCV,RidgeCV,Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


In [40]:
crop_yield = pd.read_csv("../data/crop_yield_ml")

In [41]:
crop_yield.head()

,county,year,crop_type,yield_bu_acre,PRCP,TMAX,TMIN,water_storage_0_25in,water_storage_0_50in,water_storage_0_100in,...,drainage_well,wettest_excessive,wettest_moderate,wettest_poor,wettest_somewhat_excessive,wettest_somewhat_poor,wettest_very_poor,wettest_well,soil_ph_avg,organic_matter_avg
0,Bedford,2000,CORN,108.0,4.490833,71.025,47.475000,4.025817,7.452896,13.134498,...,0.751918,0.0,0.117521,0.024722,0.04316,0.056672,0.0,0.751933,5.777047,1.057655
1,Bedford,2000,SOYBEANS,24.0,4.490833,71.025,47.475000,4.025817,7.452896,13.134498,...,0.751918,0.0,0.117521,0.024722,0.04316,0.056672,0.0,0.751933,5.777047,1.057655
2,Bedford,2000,WHEAT,46.0,4.490833,71.025,47.475000,4.025817,7.452896,13.134498,...,0.751918,0.0,0.117521,0.024722,0.04316,0.056672,0.0,0.751933,5.777047,1.057655
3,Bedford,2001,CORN,130.0,4.993333,70.675,47.933333,4.025817,7.452896,13.134498,...,0.751918,0.0,0.117521,0.024722,0.04316,0.056672,0.0,0.751933,5.777047,1.057655
4,Bedford,2001,SOYBEANS,44.0,4.993333,70.675,47.933333,4.025817,7.452896,13.134498,...,0.751918,0.0,0.117521,0.024722,0.04316,0.056672,0.0,0.751933,5.777047,1.057655


In [42]:
# Split data into train and test(unseen data)
cutoff_date = 2018

training_crop_yield_df = crop_yield[crop_yield['year'] < cutoff_date]
testing_crop_yield_df = crop_yield[crop_yield['year'] >= cutoff_date]

In [43]:
training_crop_yield_df.shape

(2816, 38)

In [44]:
testing_crop_yield_df.shape

(835, 38)

In [55]:
training_crop_yield_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2816 entries, 0 to 3641
Data columns (total 38 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   county                       2816 non-null   object 
 1   year                         2816 non-null   int64  
 2   crop_type                    2816 non-null   object 
 3   yield_bu_acre                2816 non-null   float64
 4   PRCP                         2816 non-null   float64
 5   TMAX                         2816 non-null   float64
 6   TMIN                         2816 non-null   float64
 7   water_storage_0_25in         2816 non-null   float64
 8   water_storage_0_50in         2816 non-null   float64
 9   water_storage_0_100in        2816 non-null   float64
 10  water_storage_0_150in        2816 non-null   float64
 11  typical_slope_steepness      2816 non-null   float64
 12  mean_slope_weighted          2816 non-null   float64
 13  bedrock_depth_min      

## Create Regression models

### Ridgecv regression model for CORN

In [80]:
train_crop_corn = training_crop_yield_df[training_crop_yield_df["crop_type"] == "CORN"].copy()


In [81]:
# Define X and y
target = 'yield_bu_acre'

X = train_crop_corn.drop(columns=['yield_bu_acre'])
y = train_crop_corn[target]


In [82]:
# Splitting train and test by using train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,random_state=321)

In [83]:
# Create Ridge model because Soil properties are correlated , ridge model handles correlated variables
cat_cols = X.select_dtypes(include="object").columns
num_cols = X.select_dtypes(exclude="object").columns

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])
                                 
corn_ridge_model = Pipeline([
    ("prep", preprocessor),
    ("ridge", RidgeCV(cv=5))
])

In [84]:
#fit Ridge model on the train data
corn_ridge_model.fit(X_train, y_train)
               
#generate predictions on X_test and X_train
ridge_pred_train = corn_ridge_model.predict(X_train)
ridge_y_pred_test = corn_ridge_model.predict(X_test)

#calculate mean squared error on the test and training data
ridge_mse_train = mean_squared_error(y_train, ridge_pred_train)
ridge_mse_test = mean_squared_error(y_test, ridge_y_pred_test)

print(f'corn_ridge_MSE_train: {ridge_mse_train}')
print(f'corn_ridge_MSE_test: {ridge_mse_test}')
print(f'corn_ridge_MAE_test: {mean_absolute_error(y_test, ridge_y_pred_test)}')
print(f'corn_ridge_R2_test: {r2_score(y_test, ridge_y_pred_test)}')

corn_ridge_MSE_train: 540.3093337888619
corn_ridge_MSE_test: 540.9513752407366
corn_ridge_MAE_test: 18.173874017647027
corn_ridge_R2_test: 0.3620889634907003


The Mean squared error of train corn data and test corn data is almost similar , that means the model is not overfitting.
The R2 value is 0.36 there is 36% variance , looks like model is not performing well.
Ridge model thinks if rainfall increases yield in a straight line
temperature has a fixed effect

Looks like this model is stable, but it does not capture all the factors affecting crop yields especially, non linear relationships


### Random Forest Regresion Model

In [86]:
corn_rf_model = Pipeline([
    ("prep", preprocessor),
    ("rf", RandomForestRegressor(n_estimators=200, random_state=42))
])

In [90]:
corn_rf_model.fit(X_train, y_train)
               
#generate predictions on X_test and X_train
rf_pred_train = corn_rf_model.predict(X_train)
rf_y_pred_test = corn_rf_model.predict(X_test)

#calculate mean squared error on the test and training data
rf_mse_train = mean_squared_error(y_train, rf_pred_train)
rf_mse_test = mean_squared_error(y_test, rf_y_pred_test)

print(f'rf_ridge_MSE_train: {rf_mse_train}')
print(f'rf_ridge_MSE_test: {rf_mse_test}')
print(f'rf_ridge_MAE_test: {mean_absolute_error(y_test, rf_y_pred_test)}')
print(f'rf_ridge_R2_test: {r2_score(y_test, rf_y_pred_test)}')

rf_ridge_MSE_train: 47.173972010165116
rf_ridge_MSE_test: 284.6828280769232
rf_ridge_MAE_test: 13.39651479289941
rf_ridge_R2_test: 0.6642908655992761


Compared to Ridge model the R2 value of Random forest model has significantly increased.
This means the yield dpends on nonlinear interactions.
Random forest captures soil and weather interactions.

But the mean squared error of train and test data has lot of difference.
It might be due to memorizing the patterns in training data and captures noise because of small dataset

In [ ]:
# Let's tune the model to reduce overfitting 
